Tools- model can request to call tools that can perform tasks(fetching data,searching web, running code)


Tools have:

    1.Schema- name , desc, arguments
    2.a function or co routine to execute

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model
model=init_chat_model("groq:llama-3.1-8b-instant")
res=model.invoke("what is the model name and model provider name you are using")
res.content

"I'm an AI, and my model is based on a variant of the T5 (Text-to-Text Transfer Transformer) architecture. \n\nAs for the model provider, I'm a model developed by Meta AI, which is a leading AI research organization."

In [7]:
from langchain.tools import tool

@tool
def get_weather(city:str)-> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny"

model_with_tools=model.bind_tools([get_weather])

response=model_with_tools.invoke("what is the weather in boston")
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'dy2e3hdz4', 'function': {'arguments': '{"city":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 219, 'total_tokens': 233, 'completion_time': 0.026343182, 'completion_tokens_details': None, 'prompt_time': 0.034199538, 'prompt_tokens_details': None, 'queue_time': 0.072315971, 'total_time': 0.06054272}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e37b6-f917-7cf3-b1f2-cdcfdff5918b-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'dy2e3hdz4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 14, 'total_tokens': 233})

In [13]:
import time
from langchain.agents import create_agent

@tool
def get_weather(city:str)-> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny"



agent=create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[get_weather],
    system_prompt="You are an helpful assistance."
)

response=agent.invoke({"messages":"what is the weather in boston"})
response

{'messages': [HumanMessage(content='what is the weather in boston', additional_kwargs={}, response_metadata={}, id='ad802f0e-eebe-4297-aac6-b2b14cb30455'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "boston"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e37c4-4613-7642-9aa2-ca2ac8f2fc95-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'boston'}, 'id': '55a12879-b420-45ae-9275-4b38d7c3d899', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 15, 'total_tokens': 67, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content='The weather in boston is sunny', name='get_weather', id='7f16b3fe-e0e8-4d21-89f6-831b48f3ab93', tool_call_id='55a12879-b420-45ae-9275-4b38d7c3d899'),
  AIMessage(content='The weather in Boston is sunny.', additional_kwargs={}, res

In [14]:
response["messages"][-1].content

'The weather in Boston is sunny.'

In [ ]:
messages=[{"role":"user","content":"what is the weather in sanford"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

output=model_with_tools.invoke(messages)
print(output)
print(messages)

content="Note: The output is based on the function's response as per the given context." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 250, 'total_tokens': 268, 'completion_time': 0.052653473, 'completion_tokens_details': None, 'prompt_time': 0.093776555, 'prompt_tokens_details': None, 'queue_time': 0.244963583, 'total_time': 0.146430028}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e09ee421cf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e37c4-ba10-7681-a88c-fc1f921e9d2f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 250, 'output_tokens': 18, 'total_tokens': 268}
[{'role': 'user', 'content': 'what is the weather in sanford'}, AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8gbq2hxnb', 'function': {'arguments': '{"city":"Sanford"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usa